# 04 - Modelling

This notebook trains the main data mining models: clustering, regression, classification, anomaly detection, and PCA. It loads the prepared hourly dataset created in Notebook 03.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREPARED_PATH = PROJECT_ROOT / "outputs" / "prepared_hourly_energy.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

energy_df = pd.read_csv(PREPARED_PATH, parse_dates=["datetime"])
print("Loaded prepared data:", energy_df.shape)
display(energy_df.head())

## Feature Sets

Two feature sets are used. The behavior feature set supports clustering, classification, anomaly detection, and PCA. The forecasting feature set avoids target leakage by using time and lag features for regression.

In [ ]:
behavior_features = [
    "global_reactive_power",
    "voltage",
    "global_intensity",
    "sub_metering_1",
    "sub_metering_2",
    "sub_metering_3",
    "sub_metering_total_wh",
    "unmetered_energy_wh",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

forecast_features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1_power",
    "lag_2_power",
    "lag_24_power",
    "rolling_3_power_mean",
    "rolling_24_power_mean",
]

print("Behavior features:", behavior_features)
print("Forecasting features:", forecast_features)

## Clustering: K-Means and DBSCAN

Clustering groups similar hourly consumption periods. K-Means is tested for several cluster counts, while DBSCAN is used to identify dense regions and noise points.

In [ ]:
cluster_sample = energy_df.sample(min(20000, len(energy_df)), random_state=42).sort_index()
x_cluster = cluster_sample[behavior_features].replace([np.inf, -np.inf], np.nan).dropna()
scaled_cluster = StandardScaler().fit_transform(x_cluster)

cluster_rows = []
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(scaled_cluster)
    cluster_rows.append({
        "algorithm": "K-Means",
        "clusters": k,
        "silhouette": silhouette_score(scaled_cluster, labels),
        "noise_points": np.nan,
    })

dbscan = DBSCAN(eps=1.5, min_samples=20)
dbscan_labels = dbscan.fit_predict(scaled_cluster)
dbscan_cluster_count = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
dbscan_score = silhouette_score(scaled_cluster, dbscan_labels) if dbscan_cluster_count > 1 else np.nan
cluster_rows.append({
    "algorithm": "DBSCAN",
    "clusters": dbscan_cluster_count,
    "silhouette": dbscan_score,
    "noise_points": int((dbscan_labels == -1).sum()),
})

clustering_metrics = pd.DataFrame(cluster_rows)
display(clustering_metrics)
clustering_metrics.to_csv(OUTPUT_DIR / "clustering_metrics.csv", index=False)

In [ ]:
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
clustered_df = cluster_sample.loc[x_cluster.index].copy()
clustered_df["kmeans_cluster"] = kmeans_final.fit_predict(scaled_cluster)
clustered_df["dbscan_cluster"] = dbscan_labels

pca_2d = PCA(n_components=2, random_state=42)
pca_coordinates = pca_2d.fit_transform(scaled_cluster)
clustered_df["pca_1"] = pca_coordinates[:, 0]
clustered_df["pca_2"] = pca_coordinates[:, 1]

cluster_profile = (
    clustered_df.groupby("kmeans_cluster")[["global_active_power", "global_intensity", "sub_metering_total_wh", "unmetered_energy_wh", "hour", "is_weekend"]]
    .mean()
    .round(3)
)
cluster_profile["records"] = clustered_df.groupby("kmeans_cluster").size()
display(cluster_profile)

clustered_df.to_csv(OUTPUT_DIR / "clustered_sample.csv", index=False)
cluster_profile.reset_index().to_csv(OUTPUT_DIR / "cluster_profile.csv", index=False)

## Regression: Consumption Prediction

Regression predicts hourly global active power using time and lag features. The split is chronological, so the model trains on earlier hours and tests on later hours.

In [ ]:
x_reg = energy_df[forecast_features].replace([np.inf, -np.inf], np.nan)
y_reg = energy_df["global_active_power"]
reg_df = pd.concat([x_reg, y_reg], axis=1).dropna()
x_reg = reg_df[forecast_features]
y_reg = reg_df["global_active_power"]

x_train, x_test, y_train, y_test = train_test_split(x_reg, y_reg, test_size=0.2, shuffle=False)

regression_models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Polynomial Ridge": Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)), ("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
}

regression_rows = []
for model_name, model in regression_models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    regression_rows.append({
        "model_name": model_name,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
    })

regression_metrics = pd.DataFrame(regression_rows).sort_values("rmse")
display(regression_metrics)
regression_metrics.to_csv(OUTPUT_DIR / "regression_metrics.csv", index=False)

## Classification: High vs Normal Consumption

The classification target is `high_consumption`, created from the top quartile of hourly global active power.

In [ ]:
x_cls = energy_df[behavior_features].replace([np.inf, -np.inf], np.nan)
y_cls = energy_df["high_consumption"]
cls_df = pd.concat([x_cls, y_cls], axis=1).dropna()
x_cls = cls_df[behavior_features]
y_cls = cls_df["high_consumption"]

x_train, x_test, y_train, y_test = train_test_split(x_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=42)

classification_models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
}

classification_rows = []
report_lines = []
for model_name, model in classification_models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    classification_rows.append({"model_name": model_name, "accuracy": accuracy_score(y_test, predictions)})
    report_lines.append(model_name + "\n" + classification_report(y_test, predictions, zero_division=0))

classification_metrics = pd.DataFrame(classification_rows).sort_values("accuracy", ascending=False)
display(classification_metrics)
print("\n\n".join(report_lines))

classification_metrics.to_csv(OUTPUT_DIR / "classification_metrics.csv", index=False)
(OUTPUT_DIR / "classification_report.txt").write_text("\n\n".join(report_lines), encoding="utf-8")

## Anomaly Detection and PCA

Isolation Forest marks unusual consumption periods. PCA summarizes how much feature variation is captured by a smaller number of components.

In [ ]:
anomaly_sample = energy_df.sample(min(50000, len(energy_df)), random_state=42).sort_index()
x_anomaly = anomaly_sample[behavior_features].replace([np.inf, -np.inf], np.nan).dropna()

anomaly_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", IsolationForest(contamination=0.01, random_state=42, n_estimators=150, n_jobs=-1)),
])
anomaly_labels = anomaly_pipeline.fit_predict(x_anomaly)
anomaly_scores = anomaly_pipeline.decision_function(x_anomaly)

anomaly_df = anomaly_sample.loc[x_anomaly.index].copy()
anomaly_df["anomaly_label"] = np.where(anomaly_labels == -1, 1, 0)
anomaly_df["anomaly_score"] = anomaly_scores
display(anomaly_df["anomaly_label"].value_counts().to_frame("count"))
anomaly_df.to_csv(OUTPUT_DIR / "anomaly_results.csv", index=False)

scaled_behavior = StandardScaler().fit_transform(energy_df[behavior_features].dropna())
pca = PCA(n_components=3, random_state=42)
pca.fit(scaled_behavior)
pca_summary = pd.DataFrame({
    "component": ["PC1", "PC2", "PC3"],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
})
display(pca_summary)
pca_summary.to_csv(OUTPUT_DIR / "pca_summary.csv", index=False)